In [ ]:
# %%
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

# %%
# Helper: load all monomer_i.csv files and the average.csv

# Find monomer CSVs
monomer_files = sorted(glob.glob("monomer_*.csv"))

if not monomer_files:
    raise FileNotFoundError("No 'monomer_*.csv' files found in the current directory.")

# Extract monomer index from filename using regex
monomer_pattern = re.compile(r"monomer_(\d+)\.csv")

monomer_dfs = []
for f in monomer_files:
    m = monomer_pattern.search(os.path.basename(f))
    if not m:
        continue
    monomer_id = int(m.group(1))
    df = pd.read_csv(f)
    # Expect columns: t, twist_deg, bend_deg
    if not {"t", "twist_deg", "bend_deg"}.issubset(df.columns):
        raise ValueError(f"File {f} does not have expected columns t, twist_deg, bend_deg.")
    df["monomer"] = monomer_id
    monomer_dfs.append(df)

if not monomer_dfs:
    raise RuntimeError("No valid monomer CSVs loaded.")

monomers = pd.concat(monomer_dfs, ignore_index=True).sort_values(["monomer", "t"])

# Load average.csv
avg_file = "average.csv"
if not os.path.exists(avg_file):
    raise FileNotFoundError("average.csv not found in the current directory.")

avg = pd.read_csv(avg_file)
# Expect columns: t, twist_avg_deg, bend_avg_deg
if not {"t", "twist_avg_deg", "bend_avg_deg"}.issubset(avg.columns):
    raise ValueError("average.csv does not have expected columns t, twist_avg_deg, bend_avg_deg.")

# %%
# Plot 1: Twist – all monomers in one figure

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    ax.plot(df_m["t"], df_m["twist_deg"], label=f"Monomer {monomer_id}", alpha=0.5)

ax.set_xlabel("Time")
ax.set_ylabel("Twist (deg)")
ax.set_title("Twist per monomer")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Plot 2: Bend – all monomers in one figure

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    ax.plot(df_m["t"], df_m["bend_deg"], label=f"Monomer {monomer_id}", alpha=0.5)

ax.set_xlabel("Time")
ax.set_ylabel("Bend (deg)")
ax.set_title("Bend per monomer")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Plot 3: Average twist

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(avg["t"], avg["twist_avg_deg"], color="blue",alpha = 0.5, label="Average twist")

ax.set_xlabel("Time")
ax.set_ylabel("Twist (deg)")
ax.set_title("Average twist across monomers")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# Plot 4: Average bend

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(avg["t"], avg["bend_avg_deg"], color="blue", alpha = 0.5, label="Average bend")

ax.set_xlabel("Time")
ax.set_ylabel("Bend (deg)")
ax.set_title("Average bend across monomers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# %%
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

# %%
# Load monomer_*.csv and average.csv

monomer_files = sorted(glob.glob("monomer_*.csv"))
if not monomer_files:
    raise FileNotFoundError("No 'monomer_*.csv' files found in the current directory.")

monomer_pattern = re.compile(r"monomer_(\d+)\.csv")

monomer_dfs = []
for f in monomer_files:
    m = monomer_pattern.search(os.path.basename(f))
    if not m:
        continue
    monomer_id = int(m.group(1))
    df = pd.read_csv(f)
    # Expect columns: t, twist_deg, bend_deg
    if not {"t", "twist_deg", "bend_deg"}.issubset(df.columns):
        raise ValueError(f"File {f} does not have expected columns t, twist_deg, bend_deg.")
    df["monomer"] = monomer_id
    monomer_dfs.append(df)

if not monomer_dfs:
    raise RuntimeError("No valid monomer CSVs loaded.")

monomers = pd.concat(monomer_dfs, ignore_index=True).sort_values(["monomer", "t"])

avg_file = "average.csv"
if not os.path.exists(avg_file):
    raise FileNotFoundError("average.csv not found in the current directory.")

avg = pd.read_csv(avg_file)
# Expect columns: t, twist_avg_deg, bend_avg_deg
if not {"t", "twist_avg_deg", "bend_avg_deg"}.issubset(avg.columns):
    raise ValueError("average.csv does not have expected columns t, twist_avg_deg, bend_avg_deg.")

# %%
# Histogram 1: Twist distribution per monomer

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    values = df_m["twist_deg"].values
    mean_val = np.nanmean(values)
    ax.hist(
        values,
        bins=40,
        alpha=0.4,
        label=f"Monomer {monomer_id} (mean={mean_val:.2f}°)",
        density=True,
    )

ax.set_xlabel("Twist (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Twist distribution per monomer")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Histogram 2: Bend distribution per monomer

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    values = df_m["bend_deg"].values
    mean_val = np.nanmean(values)
    ax.hist(
        values,
        bins=40,
        alpha=0.4,
        label=f"Monomer {monomer_id} (mean={mean_val:.2f}°)",
        density=True,
    )

ax.set_xlabel("Bend (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Bend distribution per monomer")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# %%

# %%
# Histogram 3: Average twist over time (with mean and std)

fig, ax = plt.subplots(figsize=(8, 4))

twist_avg_vals = avg["twist_avg_deg"].values
mean_twist_avg = np.nanmean(twist_avg_vals)
std_twist_avg = np.nanstd(twist_avg_vals)

ax.hist(
    twist_avg_vals,
    bins=40,
    alpha=0.3,
    color="tab:blue",
    density=True,
    label=f"mean = {mean_twist_avg:.2f}°, std = {std_twist_avg:.2f}°",
)

ax.set_xlabel("Average twist (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Distribution of average twist across time")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# Histogram 4: Average bend over time (with mean and std)

fig, ax = plt.subplots(figsize=(8, 4))

bend_avg_vals = avg["bend_avg_deg"].values
mean_bend_avg = np.nanmean(bend_avg_vals)
std_bend_avg = np.nanstd(bend_avg_vals)

ax.hist(
    bend_avg_vals,
    bins=40,
    alpha=0.3,
    color="tab:blue",
    density=True,
    label=f"mean = {mean_bend_avg:.2f}°, std = {std_bend_avg:.2f}°",
)

ax.set_xlabel("Average bend (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Distribution of average bend across time")
ax.legend()
plt.tight_layout()
plt.show()